In [19]:
!pip install transformers
!pip install torch
!pip install pandas

In [20]:
# Step 1: Install TextBlob
!pip install textblob

# Step 2: Import libraries
import pandas as pd
import re
from textblob import TextBlob
import warnings
warnings.filterwarnings('ignore')

# Step 3: Read raw content of the file
# Since the CSV is malformed (no clean separation), we read it as text
with open('RedditNews.csv', 'r', encoding='utf-8') as f:
    lines = f.readlines()

print("First few lines of raw data:")
print("".join(lines[:2]) + "\n")

# Step 4: Parse lines and extract (date, headline) pairs
data = []
date_pattern = r'\d{4}-\d{2}-\d{2}'  # Match YYYY-MM-DD
current_date = None

for line in lines:
    # Split line into tokens (potential dates and headlines)
    tokens = re.split(r'(?=\d{4}-\d{2}-\d{2})', line.strip())

    for token in tokens:
        if not token:
            continue

        # Extract date at the beginning of token
        match = re.match(date_pattern, token)
        if match:
            current_date = match.group()
            # The rest is the headline
            headline = token[len(current_date):].strip()
        else:
            headline = token.strip()

        # Skip if no valid date or empty headline
        if not current_date or not headline:
            continue

        data.append({'Date': current_date, 'Headline': headline})

# Step 5: Create DataFrame
df = pd.DataFrame(data)
print(f"\n Extracted {len(df)} headlines from {df['Date'].nunique()} unique days.")

# Step 6: Clean Headline column
def clean_headline(text):
    # Remove b'...' or b"..." wrapper
    text = re.sub(r"^b'(.*)'$", r"\1", text)
    text = re.sub(r'^b"(.*)"$', r'\1', text)
    # Remove escaped quotes and backslashes
    text = text.replace("\\'", "'").replace('\\"', '"').replace('\\\\', '\\')
    text = text.replace("\\r", " ").replace("\\n", " ").strip()
    return text

df['Headline'] = df['Headline'].astype(str).apply(clean_headline)

# Filter out placeholder-like or garbage
invalid = ['nan', '', 'b?', 'b', '""', "''"]
df = df[~df['Headline'].isin(invalid)]
df = df[df['Headline'].str.len() > 5]

# Step 7: Sentiment analysis function
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity  # Returns -1 to 1

print("\n Computing sentiment for each headline...")
df['Sentiment'] = df['Headline'].apply(get_sentiment)

# Step 8: Group by Date → average sentiment
daily_sentiment = df.groupby('Date')['Sentiment'].mean().reset_index()
daily_sentiment.rename(columns={'Sentiment': 'Daily_Sentiment'}, inplace=True)
daily_sentiment['Date'] = pd.to_datetime(daily_sentiment['Date'])  # Ensure proper date type

# Sort by date
daily_sentiment.sort_values('Date', inplace=True)
daily_sentiment.reset_index(drop=True, inplace=True)

# Step 9: Display results
print("\n Sample Daily Sentiment (first 10 days):")
print(daily_sentiment.head(10))

print("\n Daily Sentiment Statistics:")
print(daily_sentiment['Daily_Sentiment'].describe())

# Step 10: Save and download
output_file = 'RedditNews_Daily_Sentiment.csv'
daily_sentiment.to_csv(output_file, index=False, encoding='utf-8')
print(f"\n Saved daily sentiment to: {output_file}")

# Optional: Download in Colab
from google.colab import files
files.download(output_file)

First few lines of raw data:
Date,News
2016-07-01,"A 117-year-old woman in Mexico City finally received her birth certificate, and died a few hours later. Trinidad Alvarez Lira had waited years for proof that she had been born in 1898."



 Extracted 76105 headlines from 2943 unique days.

 Computing sentiment for each headline...

 Sample Daily Sentiment (first 10 days):
        Date  Daily_Sentiment
0 2008-06-08        -0.032892
1 2008-06-09         0.026156
2 2008-06-10        -0.029106
3 2008-06-11        -0.014500
4 2008-06-12         0.036611
5 2008-06-13         0.069818
6 2008-06-14        -0.008545
7 2008-06-15        -0.008496
8 2008-06-16         0.009005
9 2008-06-17         0.089502

 Daily Sentiment Statistics:
count    2943.000000
mean        0.012099
std         0.040681
min        -0.166995
25%        -0.014223
50%         0.012033
75%         0.038613
max         0.197128
Name: Daily_Sentiment, dtype: float64

 Saved daily sentiment to: RedditNews_Daily_Sentiment.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
from textblob import TextBlob

def get_sentiment_score(sentence):
    """
    Takes a sentence as input and returns its sentiment score.
    The score ranges from -1 (most negative) to +1 (most positive).
    """
    # Create a TextBlob object
    blob = TextBlob(sentence)

    # Get the polarity score (-1 to 1)
    sentiment_score = blob.sentiment.polarity

    return sentiment_score

# Get user input
user_sentence = input("Enter a sentence to analyze its sentiment: ")

# Calculate and display the sentiment score
score = get_sentiment_score(user_sentence)
print(f"\nSentiment score for '{user_sentence}': {score:.3f}")



Enter a sentence to analyze its sentiment: sdhjhgfd

Sentiment score for 'sdhjhgfd': 0.000


In [24]:


# Step 2: Import libraries
import pandas as pd
import re
from textblob import TextBlob
import warnings
import ipywidgets as widgets
from IPython.display import display
warnings.filterwarnings('ignore')

# Step 3: Sentiment analysis function (common for both datasets and manual input)
def get_sentiment(text):
    """Returns sentiment polarity score between -1 (negative) and 1 (positive)"""
    # Preprocessing steps
    text = str(text)
    text = re.sub(r"^b'(.*)'$", r"\1", text)  # Remove b'...' wrapper
    text = re.sub(r'^b"(.*)"$', r'\1', text)  # Remove b"..." wrapper
    text = text.replace("\\'", "'").replace('\\"', '"').replace('\\\\', '\\')  # Remove escapes
    text = text.replace("\\r", " ").replace("\\n", " ").strip()  # Clean whitespace

    # Filter out garbage text
    invalid = ['nan', '', 'b?', 'b', '""', "''"]
    if text.lower() in invalid or len(text) < 5:
        return 0.0  # Neutral score for invalid text

    return TextBlob(text).sentiment.polarity

# Step 4: Process Reddit News CSV (your original code)
def process_reddit_news():
    with open('RedditNews.csv', 'r', encoding='utf-8') as f:
        lines = f.readlines()

    data = []
    date_pattern = r'\d{4}-\d{2}-\d{2}'
    current_date = None

    for line in lines:
        tokens = re.split(r'(?=\d{4}-\d{2}-\d{2})', line.strip())
        for token in tokens:
            if not token:
                continue
            match = re.match(date_pattern, token)
            if match:
                current_date = match.group()
                headline = token[len(current_date):].strip()
            else:
                headline = token.strip()
            if current_date and headline:
                data.append({'Date': current_date, 'Headline': headline})

    df = pd.DataFrame(data)
    df['Headline'] = df['Headline'].astype(str)

    # Apply preprocessing and sentiment analysis
    df['Sentiment'] = df['Headline'].apply(get_sentiment)

    # Filter out invalid entries
    df = df[~df['Headline'].isin(['nan', '', 'b?', 'b', '""', "''"])]
    df = df[df['Headline'].str.len() > 2]

    daily_sentiment = df.groupby('Date')['Sentiment'].mean().reset_index()
    daily_sentiment.rename(columns={'Sentiment': 'Daily_Sentiment'}, inplace=True)
    daily_sentiment['Date'] = pd.to_datetime(daily_sentiment['Date'])
    daily_sentiment.sort_values('Date', inplace=True)

    output_file = 'RedditNews_Daily_Sentiment.csv'
    daily_sentiment.to_csv(output_file, index=False, encoding='utf-8')
    return daily_sentiment

# Step 5: Interactive widget for manual input
def create_sentiment_widget():
    def analyze_sentiment(b):
        input_text = text_input.value
        if not input_text.strip():
            result_label.value = "Please enter some text!"
            return

        # Apply the same preprocessing as the Reddit headlines
        processed_text = str(input_text)
        processed_text = re.sub(r"^b'(.*)'$", r"\1", processed_text)
        processed_text = re.sub(r'^b"(.*)"$', r'\1', processed_text)
        processed_text = processed_text.replace("\\'", "'").replace('\\"', '"').replace('\\\\', '\\')
        processed_text = processed_text.replace("\\r", " ").replace("\\n", " ").strip()

        score = get_sentiment(processed_text)
        result_label.value = f"Sentiment score: {score:.4f}"

    text_input = widgets.Textarea(
        placeholder="Enter your text here...",
        layout={'width': '500px', 'height': '100px'}
    )
    analyze_button = widgets.Button(
        description="Analyze Sentiment",
        button_style='info'
    )
    result_label = widgets.Label(value="")
    analyze_button.on_click(analyze_sentiment)

    display(widgets.VBox([
        widgets.Label("Enter a sentence to analyze:"),
        text_input,
        analyze_button,
        result_label
    ]))

# Step 6: Run the system
print("Reddit News Sentiment Analysis System")
print("1. Processing Reddit news data...")
reddit_sentiment = process_reddit_news()
print("Completed processing Reddit news data!")
print(f"Sample results:\n{reddit_sentiment.head()}\n")

print("2. Interactive Sentiment Analysis")
print("Enter any sentence below to get its sentiment score:")
create_sentiment_widget()

Reddit News Sentiment Analysis System
1. Processing Reddit news data...
Completed processing Reddit news data!
Sample results:
        Date  Daily_Sentiment
0 2008-06-08        -0.032892
1 2008-06-09         0.026156
2 2008-06-10        -0.029106
3 2008-06-11        -0.014500
4 2008-06-12         0.036611

2. Interactive Sentiment Analysis
Enter any sentence below to get its sentiment score:
